# Compression-BPC baseline against language-model cross-entropy: Poetry

This notebook preserves the existing poetry PDF reconstruction and stanza-preserving chunking pipeline. It scores the identical chunks with the four language models used in the cross-model study and with compression-based baselines, then compares source-level moments.

In [1]:
# Install dependencies used by the PDF parser and GPT-2 scorer.
%pip install -q pymupdf transformers accelerate scipy zstandard requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 64.6 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import re
import math
import json
import random
import traceback
import hashlib
import zipfile
from collections import Counter

import fitz
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display

DATASET_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/poet_compression_baseline")
EXTRACT_ROOT = Path("/kaggle/working/poet_dataset_extracted")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

MIN_WORDS = 200
MODEL_NAMES = {
    "gpt2": "gpt2",
    "pythia_410m": "EleutherAI/pythia-410m",
    "qwen2_5_0_5b": "Qwen/Qwen2.5-0.5B",
    "tinyllama_1_1b": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STRIDE = 512
LOWERCASE_FOR_LM = False
HISTOGRAM_METHOD = "fd"
MIN_BINS = 8
MAX_BINS = 30

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 20261012
PRE_REGISTERED_MEAN_RHO_THRESHOLD = 0.85

HEADER_FRACTION = 0.085
FOOTER_FRACTION = 0.075
STANZA_GAP_MULTIPLIER = 1.55
TITLE_UPPERCASE_RATIO = 0.82
TITLE_MAX_CHARS = 120
TITLE_MAX_WORDS = 15
TITLE_FONT_FACTOR = 1.18

NON_POEM_PATTERNS = [
    r"^source note$", r"^poems recovered", r"^working corpus",
    r"^acknowledg(?:e)?ments?$", r"^about the author$", r"^copyright",
    r"^contents$", r"^table of contents$", r"^introduction$",
    r"^foreword$", r"^notes? on the text$", r"^books by ",
    r"^also by ", r"^publisher", r"^penguin books", r"^gallery books", r"^faber",
]

TARGET_POETS = [
    "Arthur Sze", "Tracy K. Smith", "Marie Howe", "Margaret Atwood",
    "Ocean Vuong", "Rita Dove", "Kaveh Akbar", "Don Paterson",
    "A. E. Stallings", "Maria Hummel", "Patricia Lockwood", "Megan Fernandes",
    "Amanda Gorman", "Eleanor Wilner", "Sasha Debevec-McKenney", "Nick Flynn",
    "Aria Aber", "Evie Shockley", "John Lee Clark", "Kimiko Hahn", "Vona Groarke",
    "Danez Smith", "Max Ritvo", "Anselm Berrigan", "Meena Alexander",
    "Lizzie Harris", "Andrea Cohen", "Nick Laird", "Hala Alyan", "Terrance Hayes",
    "Jennifer Chang", "Frieda Hughes",
]

zip_files = sorted(DATASET_ROOT.rglob("*.zip"))
for zip_path in zip_files:
    dest = EXTRACT_ROOT / zip_path.stem
    dest.mkdir(parents=True, exist_ok=True)
    marker = dest / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(dest)
        marker.write_text("ok", encoding="utf-8")

print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_ROOT)
print("Language models:", MODEL_NAMES)
print("Device:", DEVICE)
print("Minimum chunk size:", MIN_WORDS, "words")

Dataset: /kaggle/input
Output: /kaggle/working/poet_compression_baseline
Language models: {'gpt2': 'gpt2', 'pythia_410m': 'EleutherAI/pythia-410m', 'qwen2_5_0_5b': 'Qwen/Qwen2.5-0.5B', 'tinyllama_1_1b': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}
Device: cuda
Minimum chunk size: 200 words


In [3]:
import os
import re
import math
import json
import random
import traceback
from collections import Counter

import fitz
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyMuPDF loaded.")
print("Transformers loaded.")

PyMuPDF loaded.
Transformers loaded.


## 1. Find poet PDFs

In [4]:
def normalize_name(s):
    s = s.lower().replace("_", " ").replace("-", " ")
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

normalized_poets = {
    normalize_name(p): p
    for p in TARGET_POETS
}

def identify_poet(path):
    stem = normalize_name(path.stem)
    matches = [
        poet
        for norm, poet in normalized_poets.items()
        if norm in stem
    ]
    return matches[0] if len(matches) == 1 else None

pdf_candidates = set(DATASET_ROOT.rglob("*.pdf")) | set(EXTRACT_ROOT.rglob("*.pdf"))
pdf_files = sorted((p.resolve() for p in pdf_candidates), key=lambda p: str(p))

discovery_rows = []
poet_to_paths = {}

for path in pdf_files:
    poet = identify_poet(path)

    discovery_rows.append({
        "file": str(path),
        "poet": poet,
        "recognized": poet is not None,
    })

    if poet is not None:
        poet_to_paths.setdefault(poet, []).append(path)

discovery_df = pd.DataFrame(discovery_rows)

print("PDFs discovered:", len(pdf_files))
print(
    "Recognized target-poet PDFs:",
    int(discovery_df["recognized"].sum())
)

for poet in TARGET_POETS:
    paths = poet_to_paths.get(poet, [])
    print(f"{poet:30s} -> {len(paths)} file(s)")
    for p in paths:
        print("   ", p)

discovery_df.to_csv(
    OUTPUT_ROOT / "discovery_report.csv",
    index=False,
    encoding="utf-8-sig",
)

PDFs discovered: 32
Recognized target-poet PDFs: 31
Arthur Sze                     -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/01_arthur_sze.pdf
Tracy K. Smith                 -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/02_tracy_k_smith.pdf
Marie Howe                     -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/03_marie_howe.pdf
Margaret Atwood                -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/04_margaret_atwood.pdf
Ocean Vuong                    -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/05_ocean_vuong.pdf
Rita Dove                      -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/06_rita_dove.pdf
Kaveh Akbar                    -> 1 file(s)
    /kaggle/input/datasets/srabongitik

## 2. Reconstruct poems and stanzas from PDF layout

In [5]:
compiled_non_poem = [
    re.compile(pattern, re.I)
    for pattern in NON_POEM_PATTERNS
]

def normalize_spaces(text):
    return re.sub(r"[ \t]+", " ", text.strip())

def is_roman_label(text):
    return bool(
        re.fullmatch(r"[IVXLCDM]+", text.strip(), flags=re.I)
    )

def is_non_poem_line(text):
    t = normalize_spaces(text)

    if not t:
        return True

    if is_roman_label(t):
        return True

    if re.fullmatch(r"\d{1,4}", t):
        return True

    return any(rx.search(t) for rx in compiled_non_poem)

def uppercase_ratio(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(c.isupper() for c in letters) / len(letters)

def count_words(text):
    return len(re.findall(
        r"(?u)\b[\wÀ-ÖØ-öø-ÿ]+(?:['’\-][\wÀ-ÖØ-öø-ÿ]+)*\b",
        text
    ))

def extract_page_lines(page, page_number):
    data = page.get_text("dict")
    lines = []

    for block in data.get("blocks", []):
        if block.get("type") != 0:
            continue

        for line in block.get("lines", []):
            spans = [
                sp
                for sp in line.get("spans", [])
                if sp.get("text", "").strip()
            ]

            if not spans:
                continue

            spans.sort(key=lambda sp: sp["bbox"][0])

            text = "".join(
                sp["text"]
                for sp in spans
            ).strip()

            if not text:
                continue

            x0 = min(sp["bbox"][0] for sp in spans)
            y0 = min(sp["bbox"][1] for sp in spans)
            x1 = max(sp["bbox"][2] for sp in spans)
            y1 = max(sp["bbox"][3] for sp in spans)

            font_size = float(np.median([
                float(sp.get("size", 10.0))
                for sp in spans
            ]))

            lines.append({
                "page": page_number,
                "text": text,
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "font_size": font_size,
                "page_height": page.rect.height,
                "page_width": page.rect.width,
            })

    lines.sort(
        key=lambda x: (x["y0"], x["x0"])
    )

    return lines

def looks_like_title(
    text,
    font_size,
    typical_body_font,
):
    t = normalize_spaces(text)

    if not t or is_non_poem_line(t):
        return False

    if len(t) > TITLE_MAX_CHARS:
        return False

    if count_words(t) > TITLE_MAX_WORDS:
        return False

    if sum(c.isalpha() for c in t) < 2:
        return False

    if uppercase_ratio(t) >= TITLE_UPPERCASE_RATIO:
        return True

    if (
        typical_body_font
        and font_size
        >= TITLE_FONT_FACTOR * typical_body_font
    ):
        return True

    return False

def build_poems(pdf_path):
    doc = fitz.open(str(pdf_path))

    all_pages = []
    body_sizes = []

    for page_number, page in enumerate(doc, start=1):
        lines = extract_page_lines(
            page,
            page_number,
        )

        kept = []

        for line in lines:
            relative_y = (
                line["y0"]
                / max(line["page_height"], 1)
            )

            if relative_y < HEADER_FRACTION:
                continue

            if relative_y > 1.0 - FOOTER_FRACTION:
                continue

            if is_non_poem_line(line["text"]):
                continue

            kept.append(line)

            if (
                count_words(line["text"]) > 0
                and uppercase_ratio(line["text"]) < 0.95
            ):
                body_sizes.append(line["font_size"])

        all_pages.append(kept)

    doc.close()

    typical_body_font = (
        float(np.median(body_sizes))
        if body_sizes
        else 10.0
    )

    events = []

    for page_lines in all_pages:
        gaps = [
            b["y0"] - a["y1"]
            for a, b
            in zip(page_lines, page_lines[1:])
            if b["y0"] >= a["y1"]
        ]

        base_gap = (
            float(np.median(gaps))
            if gaps
            else 4.0
        )

        for line in page_lines:
            events.append({
                "kind": (
                    "title"
                    if looks_like_title(
                        line["text"],
                        line["font_size"],
                        typical_body_font,
                    )
                    else "body"
                ),
                "line": line,
                "base_gap": base_gap,
            })

    poems = []
    current = None

    def finalize():
        nonlocal current

        if current is None:
            return

        if current["current_stanza"]:
            current["stanzas"].append(
                current["current_stanza"]
            )
            current["current_stanza"] = []

        if current["stanzas"]:
            poems.append({
                "title": current["title"],
                "start_page": current["start_page"],
                "end_page": current["end_page"],
                "stanzas": current["stanzas"],
            })

        current = None

    for event in events:
        line = event["line"]

        if event["kind"] == "title":
            finalize()

            current = {
                "title": normalize_spaces(
                    line["text"]
                ),
                "start_page": line["page"],
                "end_page": line["page"],
                "stanzas": [],
                "current_stanza": [],
                "base_gap": event["base_gap"],
                "previous_line": None,
            }

            continue

        if current is None:
            continue

        stanza_break = False
        previous = current["previous_line"]

        if (
            previous is not None
            and previous["page"] == line["page"]
        ):
            gap = line["y0"] - previous["y1"]

            if gap > (
                current["base_gap"]
                * STANZA_GAP_MULTIPLIER
            ):
                stanza_break = True

        if stanza_break and current["current_stanza"]:
            current["stanzas"].append(
                current["current_stanza"]
            )
            current["current_stanza"] = []

        current["current_stanza"].append(line)
        current["end_page"] = line["page"]
        current["previous_line"] = line

    finalize()

    return poems

## 3. Create stanza-preserving chunks

In [6]:
def stanza_text(stanza):
    return "\n".join(
        line["text"].rstrip()
        for line in stanza
    )

def create_chunks(poems, min_words):
    chunks = []

    for poem_index, poem in enumerate(
        poems,
        start=1,
    ):
        stanza_rows = []

        for stanza_index, stanza in enumerate(
            poem["stanzas"],
            start=1,
        ):
            text = stanza_text(stanza)

            stanza_rows.append({
                "stanza_index": stanza_index,
                "text": text,
                "word_count": count_words(text),
                "page_start": stanza[0]["page"],
                "page_end": stanza[-1]["page"],
            })

        if not stanza_rows:
            continue

        poem_word_count = sum(
            row["word_count"]
            for row in stanza_rows
        )

        # Entire short poem = one chunk.
        if poem_word_count < min_words:
            chunks.append({
                "poem_index": poem_index,
                "chunk_within_poem": 1,
                "poem_title": poem["title"],
                "poem_start_page": poem["start_page"],
                "poem_end_page": poem["end_page"],
                "stanza_start": 1,
                "stanza_end": len(stanza_rows),
                "stanza_count": len(stanza_rows),
                "word_count": poem_word_count,
                "text": "\n\n".join(
                    row["text"]
                    for row in stanza_rows
                ),
            })
            continue

        current = []
        current_words = 0
        poem_chunks = []

        for stanza in stanza_rows:
            current.append(stanza)
            current_words += stanza["word_count"]

            if current_words >= min_words:
                poem_chunks.append({
                    "poem_index": poem_index,
                    "chunk_within_poem": (
                        len(poem_chunks) + 1
                    ),
                    "poem_title": poem["title"],
                    "poem_start_page": poem["start_page"],
                    "poem_end_page": poem["end_page"],
                    "stanza_start": (
                        current[0]["stanza_index"]
                    ),
                    "stanza_end": (
                        current[-1]["stanza_index"]
                    ),
                    "stanza_count": len(current),
                    "word_count": current_words,
                    "text": "\n\n".join(
                        row["text"]
                        for row in current
                    ),
                })

                current = []
                current_words = 0

        # Do not create an undersized final chunk for a long poem.
        if current:
            remainder_text = "\n\n".join(
                row["text"]
                for row in current
            )

            if (
                poem_chunks
                and current_words < min_words
            ):
                previous = poem_chunks[-1]

                previous["stanza_end"] = (
                    current[-1]["stanza_index"]
                )
                previous["stanza_count"] += len(current)
                previous["word_count"] += current_words
                previous["text"] += (
                    "\n\n" + remainder_text
                )
            else:
                poem_chunks.append({
                    "poem_index": poem_index,
                    "chunk_within_poem": (
                        len(poem_chunks) + 1
                    ),
                    "poem_title": poem["title"],
                    "poem_start_page": poem["start_page"],
                    "poem_end_page": poem["end_page"],
                    "stanza_start": (
                        current[0]["stanza_index"]
                    ),
                    "stanza_end": (
                        current[-1]["stanza_index"]
                    ),
                    "stanza_count": len(current),
                    "word_count": current_words,
                    "text": remainder_text,
                })

        chunks.extend(poem_chunks)

    for chunk_id, chunk in enumerate(
        chunks,
        start=1,
    ):
        chunk["chunk_id"] = chunk_id

    return chunks

## 4. Parse all available poets

In [7]:
poet_chunks = {}
source_status = []

for poet in TARGET_POETS:
    matched_paths = poet_to_paths.get(poet, [])
    if not matched_paths:
        source_status.append({"source_name": poet, "source_pdf": "", "status": "NO_PDF_FOUND", "n_poems": 0, "n_chunks": 0, "error": ""})
        print(f"[MISSING] {poet}: no matching PDF")
        continue

    pdf_path = matched_paths[0]
    try:
        poems = build_poems(pdf_path)
        chunks = create_chunks(poems, MIN_WORDS)
        if not chunks:
            raise RuntimeError("No chunks were created by the poetry parser.")
        poet_chunks[poet] = {"pdf_path": str(pdf_path), "poems": poems, "chunks": chunks}
        source_status.append({"source_name": poet, "source_pdf": str(pdf_path), "status": "READY", "n_poems": len(poems), "n_chunks": len(chunks), "error": ""})
        print(f"{poet}: {len(poems)} poems, {len(chunks)} chunks")
        if len(matched_paths) > 1:
            print(f"  Multiple files matched; using {pdf_path.name}")
    except Exception as exc:
        source_status.append({"source_name": poet, "source_pdf": str(pdf_path), "status": "PARSER_ERROR", "n_poems": 0, "n_chunks": 0, "error": f"{type(exc).__name__}: {exc}"})
        print(f"[ERROR] {poet}: {type(exc).__name__}: {exc}")

source_status_df = pd.DataFrame(source_status)
source_status_df.to_csv(OUTPUT_ROOT / "poet_run_status_gpt2.csv", index=False, encoding="utf-8-sig")
print("Sources ready:", len(poet_chunks), "of", len(TARGET_POETS))

Arthur Sze: 1 poems, 1 chunks
Tracy K. Smith: 126 poems, 146 chunks
Marie Howe: 32 poems, 34 chunks
Margaret Atwood: 17 poems, 217 chunks
Ocean Vuong: 26 poems, 33 chunks
Rita Dove: 110 poems, 115 chunks
Kaveh Akbar: 59 poems, 61 chunks
Don Paterson: 81 poems, 107 chunks
[MISSING] A. E. Stallings: no matching PDF
Maria Hummel: 32 poems, 33 chunks
Patricia Lockwood: 24 poems, 41 chunks
Megan Fernandes: 58 poems, 58 chunks
Amanda Gorman: 129 poems, 146 chunks
Eleanor Wilner: 110 poems, 132 chunks
Sasha Debevec-McKenney: 39 poems, 54 chunks
Nick Flynn: 1 poems, 1 chunks
Aria Aber: 90 poems, 103 chunks
Evie Shockley: 27 poems, 59 chunks
John Lee Clark: 43 poems, 46 chunks
Kimiko Hahn: 122 poems, 162 chunks
Vona Groarke: 49 poems, 56 chunks
Danez Smith: 15 poems, 58 chunks
Max Ritvo: 33 poems, 33 chunks
Anselm Berrigan: 19 poems, 34 chunks
Meena Alexander: 39 poems, 46 chunks
Lizzie Harris: 1 poems, 1 chunks
Andrea Cohen: 69 poems, 70 chunks
Nick Laird: 27 poems, 36 chunks
Hala Alyan: 41 po

## 4. Teacher-forced LM scoring helper

In [8]:
import torch
import torch.nn.functional as F

def score_text_lm(text):
    text_for_model = text.lower() if LOWERCASE_FOR_LM else text
    char_count = len(text_for_model)
    input_ids = tokenizer(
        text_for_model,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(DEVICE)
    sequence_length = input_ids.size(1)

    if sequence_length < 2 or char_count == 0:
        return {
            "total_bits": np.nan,
            "scored_tokens": 0,
            "bits_per_token": np.nan,
            "bpc": np.nan,
            "model_tokens": sequence_length,
            "character_count": char_count,
        }

    stride = min(STRIDE, max(1, MODEL_CONTEXT - 1))
    total_nll_nats = 0.0
    scored_tokens = 0
    previous_end = 0

    with torch.inference_mode():
        for end in range(min(stride, sequence_length), sequence_length + stride, stride):
            end = min(end, sequence_length)
            begin = max(0, end - MODEL_CONTEXT)
            input_slice = input_ids[:, begin:end]
            target_ids = input_slice.clone()
            newly_scored = end - previous_end
            if newly_scored < input_slice.size(1):
                target_ids[:, :-newly_scored] = -100

            outputs = model(input_ids=input_slice)
            logits = outputs.logits[:, :-1, :].contiguous()
            labels = target_ids[:, 1:].contiguous()
            loss_sum = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
                reduction="sum",
            )
            total_nll_nats += float(loss_sum.item())
            scored_tokens += int((labels != -100).sum().item())
            previous_end = end
            if end >= sequence_length:
                break

    total_bits = total_nll_nats / math.log(2)
    return {
        "total_bits": total_bits,
        "scored_tokens": scored_tokens,
        "bits_per_token": total_bits / scored_tokens if scored_tokens else np.nan,
        "bpc": total_bits / char_count,
        "model_tokens": sequence_length,
        "character_count": char_count,
    }

## 5. Fixed compression reference and compressor definitions

In [9]:
import bz2
import gc
import hashlib
import lzma
import math
import os
import re
import time
import zlib
from pathlib import Path

import requests
import zstandard as zstd
from scipy.stats import spearmanr, pearsonr, rankdata

REFERENCE_URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"
REFERENCE_NAME = "The Adventures of Sherlock Holmes (Project Gutenberg eBook 1661)"
REFERENCE_CHAR_LIMIT = 80000
REFERENCE_CACHE_PATH = OUTPUT_ROOT / "neutral_english_reference.txt"

def load_neutral_reference():
    if REFERENCE_CACHE_PATH.exists():
        raw = REFERENCE_CACHE_PATH.read_text(encoding="utf-8", errors="replace")
    else:
        response = requests.get(REFERENCE_URL, timeout=90)
        response.raise_for_status()
        raw = response.text
        start = re.search(
            r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*",
            raw, flags=re.IGNORECASE
        )
        if start:
            raw = raw[start.end():]
        end = re.search(
            r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*",
            raw, flags=re.IGNORECASE
        )
        if end:
            raw = raw[:end.start()]
        raw = raw.strip()
        REFERENCE_CACHE_PATH.write_text(raw[:REFERENCE_CHAR_LIMIT], encoding="utf-8")
        raw = raw[:REFERENCE_CHAR_LIMIT]
    if len(raw) < 50000:
        raise RuntimeError(
            f"Reference text has only {len(raw)} characters. Check download/cache: {REFERENCE_CACHE_PATH}"
        )
    return raw[:REFERENCE_CHAR_LIMIT]

REFERENCE_TEXT = load_neutral_reference()
REFERENCE_BYTES = REFERENCE_TEXT.encode("utf-8")
print(f"Fixed compression reference: {REFERENCE_NAME}")
print(f"Reference: {REFERENCE_URL}")
print(f"Reference length: {len(REFERENCE_TEXT):,} characters; selected before examining results.")

def c_deflate(data: bytes) -> int:
    comp = zlib.compressobj(level=9, wbits=-15)
    return len(comp.compress(data) + comp.flush())

def c_lzma(data: bytes) -> int:
    filters = [{"id": lzma.FILTER_LZMA2, "preset": 9 | lzma.PRESET_EXTREME}]
    return len(lzma.compress(data, format=lzma.FORMAT_RAW, filters=filters))

def c_bz2(data: bytes) -> int:
    return len(bz2.compress(data, compresslevel=9))

ZSTD_COMPRESSOR = zstd.ZstdCompressor(level=19)

def c_zstd(data: bytes) -> int:
    return len(ZSTD_COMPRESSOR.compress(data))

COMPRESSORS = {
    "plain_deflate": c_deflate,
    "plain_lzma": c_lzma,
    "plain_bz2": c_bz2,
    "plain_zstd": c_zstd,
}

PRIMED_COMPRESSORS = {
    "primed_deflate": c_deflate,
    "primed_lzma": c_lzma,
}
REFERENCE_COMPRESSED_SIZES = {
    method: comp(REFERENCE_BYTES)
    for method, comp in PRIMED_COMPRESSORS.items()
}

def compression_bpc(text: str, method: str) -> float:
    n_chars = len(text)
    if n_chars == 0:
        return np.nan
    data = text.encode("utf-8")
    if method in COMPRESSORS:
        compressed_size = COMPRESSORS[method](data)
    elif method in PRIMED_COMPRESSORS:
        compressed_size = PRIMED_COMPRESSORS[method](REFERENCE_BYTES + data)
        compressed_size -= REFERENCE_COMPRESSED_SIZES[method]
    else:
        raise KeyError(f"Unknown compressor method: {method}")
    # Character denominator matches the study's BPC definition; do not use byte length.
    return 8.0 * compressed_size / n_chars

COMPRESSION_METHODS = [
    "plain_deflate", "plain_lzma", "plain_bz2", "plain_zstd",
    "primed_deflate", "primed_lzma",
]
METHOD_KIND = {
    **{key: "language_model" for key in MODEL_NAMES},
    **{key: "compressor" for key in COMPRESSION_METHODS},
}
MEASURES = ["mean_bpc", "variance_bpc", "skewness_bpc", "excess_kurtosis_bpc"]

Fixed compression reference: The Adventures of Sherlock Holmes (Project Gutenberg eBook 1661)
Reference: https://www.gutenberg.org/cache/epub/1661/pg1661.txt
Reference length: 80,000 characters; selected before examining results.


## 6. Build the analysis chunk manifest

In [10]:
CORPUS_PREFIX = "poet"
SOURCE_CHUNKS = poet_chunks

In [11]:
chunk_records = []
chunking_status_rows = []

for source_name, payload in SOURCE_CHUNKS.items():
    chunks = payload["chunks"]
    if not chunks:
        chunking_status_rows.append({
            "source_id": source_name, "source_name": source_name,
            "status": "NO_CHUNKS", "n_chunks": 0, "error": ""
        })
        continue

    for idx, chunk in enumerate(chunks, start=1):
        text = chunk.get("text", "")
        chunk_records.append({
            "source_id": source_name,
            "source_name": source_name,
            "chunk_id": chunk.get("chunk_id", idx),
            "text": text,
            "word_count": int(chunk.get("word_count", count_words(text))),
            "character_count": len(text),
            "source_pdf": payload.get("pdf_path", ""),
            "poem_index": chunk.get("poem_index", np.nan),
            "poem_title": chunk.get("poem_title", ""),
            "section_index": chunk.get("section_index", np.nan),
            "section_heading": chunk.get("section_heading", ""),
            "start_page": chunk.get("start_page", chunk.get("poem_start_page", np.nan)),
            "end_page": chunk.get("end_page", chunk.get("poem_end_page", np.nan)),
        })
    chunking_status_rows.append({
        "source_id": source_name, "source_name": source_name,
        "status": "READY", "n_chunks": len(chunks), "error": ""
    })

chunk_manifest_df = pd.DataFrame(chunk_records)
chunking_status_df = pd.DataFrame(chunking_status_rows)
chunk_manifest_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_chunk_manifest.csv"
chunking_status_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_chunking_status.csv"
chunk_manifest_df.drop(columns=["text"]).to_csv(chunk_manifest_path, index=False, encoding="utf-8-sig")
chunking_status_df.to_csv(chunking_status_path, index=False, encoding="utf-8-sig")

if chunk_manifest_df.empty:
    raise RuntimeError("No chunks were reconstructed; check the dataset and chunking logs.")

print(f"Reconstructed {len(chunk_manifest_df)} chunks from {chunk_manifest_df['source_id'].nunique()} sources.")
print("Chunk manifest:", chunk_manifest_path)

Reconstructed 2140 chunks from 31 sources.
Chunk manifest: /kaggle/working/poet_compression_baseline/poet_chunk_manifest.csv


## 7. Compute compression-BPC baselines

In [12]:
compression_chunk_rows = []

for row in chunk_records:
    base = {
        "source_id": row["source_id"],
        "source_name": row["source_name"],
        "chunk_id": row["chunk_id"],
        "character_count": len(row["text"]),
        "word_count": row["word_count"],
    }
    for method in COMPRESSION_METHODS:
        try:
            value = compression_bpc(row["text"], method)
        except Exception as exc:
            print(f"[COMPRESSION ERROR] {row['source_name']} chunk {row['chunk_id']} / {method}: {exc}")
            value = np.nan
        compression_chunk_rows.append({
            **base,
            "method": method,
            "bpc": value,
            "compressor_status": "SUCCESS" if np.isfinite(value) else "ERROR_OR_NONFINITE",
        })

compression_chunk_df = pd.DataFrame(compression_chunk_rows)
compression_chunk_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_compression_chunk_scores.csv"
compression_chunk_df.to_csv(compression_chunk_path, index=False, encoding="utf-8-sig")
print("Compression chunk scores saved:", compression_chunk_path)
print("Compression methods:", ", ".join(COMPRESSION_METHODS))

Compression chunk scores saved: /kaggle/working/poet_compression_baseline/poet_compression_chunk_scores.csv
Compression methods: plain_deflate, plain_lzma, plain_bz2, plain_zstd, primed_deflate, primed_lzma


## 8. Score the same chunks with the four language models

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def infer_model_context_length(tokenizer, model):
    candidates = [
        getattr(tokenizer, "model_max_length", None),
        getattr(model.config, "n_positions", None),
        getattr(model.config, "max_position_embeddings", None),
        getattr(model.config, "max_sequence_length", None),
    ]
    valid = [int(x) for x in candidates if isinstance(x, int) and 1 < x < 100000]
    return min(valid) if valid else 1024

lm_chunk_rows = []
model_status_rows = []

for model_key, model_name in MODEL_NAMES.items():
    model_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_lm_chunk_scores_{model_key}.csv"
    print(f"\n=== Loading {model_key}: {model_name} ===")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            low_cpu_mem_usage=True,
        )
        model.to(DEVICE)
        model.eval()
        if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
        MODEL_CONTEXT = infer_model_context_length(tokenizer, model)
        print("Context length:", MODEL_CONTEXT, "| Device:", DEVICE)

        this_model_rows = []
        for count, row in enumerate(chunk_records, start=1):
            try:
                score = score_text_lm(row["text"])
                this_model_rows.append({
                    "source_id": row["source_id"],
                    "source_name": row["source_name"],
                    "chunk_id": row["chunk_id"],
                    "method": model_key,
                    "model_name": model_name,
                    "bpc": score["bpc"],
                    "bits_per_token": score["bits_per_token"],
                    "total_bits": score["total_bits"],
                    "scored_tokens": score["scored_tokens"],
                    "character_count": score["character_count"],
                    "word_count": row["word_count"],
                    "context_length": MODEL_CONTEXT,
                })
            except Exception as exc:
                this_model_rows.append({
                    "source_id": row["source_id"],
                    "source_name": row["source_name"],
                    "chunk_id": row["chunk_id"],
                    "method": model_key,
                    "model_name": model_name,
                    "bpc": np.nan,
                    "bits_per_token": np.nan,
                    "total_bits": np.nan,
                    "scored_tokens": 0,
                    "character_count": len(row["text"]),
                    "word_count": row["word_count"],
                    "context_length": MODEL_CONTEXT,
                    "error": f"{type(exc).__name__}: {exc}",
                })
            if count % 100 == 0:
                print(f"  Scored {count}/{len(chunk_records)} chunks")

        model_df = pd.DataFrame(this_model_rows)
        model_df.to_csv(model_path, index=False, encoding="utf-8-sig")
        lm_chunk_rows.extend(this_model_rows)
        model_status_rows.append({
            "method": model_key,
            "model_name": model_name,
            "status": "SUCCESS",
            "n_chunks": len(model_df),
            "n_valid_bpc": int(np.isfinite(pd.to_numeric(model_df["bpc"], errors="coerce")).sum()),
            "context_length": MODEL_CONTEXT,
            "error": "",
        })
        print("Saved:", model_path)

        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:
        model_status_rows.append({
            "method": model_key,
            "model_name": model_name,
            "status": "FAILED",
            "n_chunks": 0,
            "n_valid_bpc": 0,
            "context_length": np.nan,
            "error": f"{type(exc).__name__}: {exc}",
        })
        print(f"[MODEL FAILED] {model_key}: {type(exc).__name__}: {exc}")
        try:
            del model, tokenizer
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

lm_chunk_df = pd.DataFrame(lm_chunk_rows)
if not lm_chunk_df.empty:
    lm_chunk_df.to_csv(OUTPUT_ROOT / f"{CORPUS_PREFIX}_lm_chunk_scores_all_models.csv", index=False, encoding="utf-8-sig")
else:
    raise RuntimeError("No language-model chunk scores were produced. Check model access, network, and the model_run_status output.")
model_status_df = pd.DataFrame(model_status_rows)
model_status_df.to_csv(OUTPUT_ROOT / f"{CORPUS_PREFIX}_model_run_status.csv", index=False, encoding="utf-8-sig")
display(model_status_df)

successful_lms = [
    k for k in MODEL_NAMES
    if k in set(lm_chunk_df["method"]) and
       lm_chunk_df.loc[lm_chunk_df["method"].eq(k), "bpc"].notna().any()
]
if len(successful_lms) < 2:
    raise RuntimeError("Fewer than two language models produced valid chunk scores; cross-model comparison cannot be computed.")
print("Successful language models:", successful_lms)


=== Loading gpt2: gpt2 ===


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Context length: 2048 | Device: cuda
  Scored 100/2140 chunks
  Scored 200/2140 chunks
  Scored 300/2140 chunks
  Scored 400/2140 chunks
  Scored 500/2140 chunks
  Scored 600/2140 chunks
  Scored 700/2140 chunks
  Scored 800/2140 chunks
  Scored 900/2140 chunks
  Scored 1000/2140 chunks
  Scored 1100/2140 chunks


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2071 > 2048). Running this sequence through the model will result in indexing errors


  Scored 1200/2140 chunks
  Scored 1300/2140 chunks
  Scored 1400/2140 chunks
  Scored 1500/2140 chunks
  Scored 1600/2140 chunks
  Scored 1700/2140 chunks
  Scored 1800/2140 chunks
  Scored 1900/2140 chunks
  Scored 2000/2140 chunks
  Scored 2100/2140 chunks
Saved: /kaggle/working/poet_compression_baseline/poet_lm_chunk_scores_tinyllama_1_1b.csv


,method,model_name,status,n_chunks,n_valid_bpc,context_length,error
0,gpt2,gpt2,SUCCESS,2140,2135,1024,
1,pythia_410m,EleutherAI/pythia-410m,SUCCESS,2140,2135,2048,
2,qwen2_5_0_5b,Qwen/Qwen2.5-0.5B,SUCCESS,2140,2134,32768,
3,tinyllama_1_1b,TinyLlama/TinyLlama-1.1B-Chat-v1.0,SUCCESS,2140,2135,2048,


Successful language models: ['gpt2', 'pythia_410m', 'qwen2_5_0_5b', 'tinyllama_1_1b']


## 9. Calculate raw and binned source-level moments

In [14]:
from scipy.stats import skew, kurtosis

METHOD_NAMES = {
    "gpt2": "GPT-2",
    "pythia_410m": "Pythia-410M",
    "qwen2_5_0_5b": "Qwen2.5-0.5B",
    "tinyllama_1_1b": "TinyLlama-1.1B",
    "plain_deflate": "Raw Deflate (level 9)",
    "plain_lzma": "Raw LZMA2 (level 9 extreme)",
    "plain_bz2": "BZip2 (level 9)",
    "plain_zstd": "Zstandard (level 19)",
    "primed_deflate": "Primed raw Deflate",
    "primed_lzma": "Primed raw LZMA2",
}

all_score_rows = []
if not lm_chunk_df.empty:
    all_score_rows.extend(lm_chunk_df.to_dict("records"))
all_score_rows.extend(compression_chunk_df.to_dict("records"))
all_chunk_score_df = pd.DataFrame(all_score_rows)
all_chunk_score_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_all_chunk_scores_lm_and_compressors.csv"
all_chunk_score_df.to_csv(all_chunk_score_path, index=False, encoding="utf-8-sig")

source_meta = (
    chunk_manifest_df.groupby(["source_id", "source_name"], dropna=False)
    .agg(n_chunks=("chunk_id", "count"),
         mean_chunk_length_chars=("character_count", "mean"))
    .reset_index()
)

def histogram_frequency(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return pd.DataFrame(columns=["bin_left", "bin_right", "bin_center", "frequency"])
    if len(values) == 1 or np.isclose(values.min(), values.max()):
        center = float(values[0])
        edges = np.array([center - 0.5, center + 0.5], dtype=float)
    else:
        fd_edges = np.histogram_bin_edges(values, bins="fd")
        suggested = max(1, len(fd_edges) - 1)
        n_bins = int(np.clip(suggested, MIN_BINS, MAX_BINS))
        edges = np.linspace(float(values.min()), float(values.max()), n_bins + 1)
    counts, edges = np.histogram(values, bins=edges)
    centers = (edges[:-1] + edges[1:]) / 2
    keep = counts > 0
    return pd.DataFrame({
        "bin_left": edges[:-1][keep],
        "bin_right": edges[1:][keep],
        "bin_center": centers[keep],
        "frequency": counts[keep].astype(int),
    })

def moments_from_values(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n == 0:
        return {"n_valid_chunks": 0, "n_histogram_bins": 0,
                "mean_bpc": np.nan, "variance_bpc": np.nan,
                "skewness_bpc": np.nan, "excess_kurtosis_bpc": np.nan,
                "third_central_moment_bpc": np.nan, "fourth_central_moment_bpc": np.nan}
    mean = float(np.mean(values))
    centered = values - mean
    m2 = float(np.mean(centered ** 2))
    m3 = float(np.mean(centered ** 3))
    m4 = float(np.mean(centered ** 4))
    skew_value = m3 / (m2 ** 1.5) if m2 > 0 else np.nan
    excess = m4 / (m2 ** 2) - 3.0 if m2 > 0 else np.nan
    return {
        "n_valid_chunks": n, "n_histogram_bins": np.nan,
        "mean_bpc": mean, "variance_bpc": m2,
        "skewness_bpc": float(skew_value) if np.isfinite(skew_value) else np.nan,
        "excess_kurtosis_bpc": float(excess) if np.isfinite(excess) else np.nan,
        "third_central_moment_bpc": m3,
        "fourth_central_moment_bpc": m4,
    }

def moments_from_histogram(hist):
    if hist.empty:
        return moments_from_values([])
    centers = hist["bin_center"].to_numpy(dtype=float)
    weights = hist["frequency"].to_numpy(dtype=float)
    n = int(weights.sum())
    mean = float(np.sum(weights * centers) / n)
    centered = centers - mean
    m2 = float(np.sum(weights * centered**2) / n)
    m3 = float(np.sum(weights * centered**3) / n)
    m4 = float(np.sum(weights * centered**4) / n)
    return {
        "n_valid_chunks": n, "n_histogram_bins": int(len(hist)),
        "mean_bpc": mean, "variance_bpc": m2,
        "skewness_bpc": float(m3 / m2**1.5) if m2 > 0 else np.nan,
        "excess_kurtosis_bpc": float(m4 / m2**2 - 3.0) if m2 > 0 else np.nan,
        "third_central_moment_bpc": m3,
        "fourth_central_moment_bpc": m4,
    }

method_values = {
    method: (
        all_chunk_score_df.loc[all_chunk_score_df["method"].eq(method)]
        .groupby("source_id")["bpc"]
        .apply(lambda s: pd.to_numeric(s, errors="coerce").to_numpy(dtype=float))
        .to_dict()
    )
    for method in all_chunk_score_df["method"].dropna().unique()
}

moment_rows = []
frequency_rows = []
for src in source_meta.itertuples(index=False):
    for method, by_source in method_values.items():
        values = np.asarray(by_source.get(src.source_id, []), dtype=float)
        values = values[np.isfinite(values)]
        raw_stats = moments_from_values(values)
        common = {
            "source_id": src.source_id,
            "source_name": src.source_name,
            "method": method,
            "method_name": METHOD_NAMES.get(method, method),
            "method_type": METHOD_KIND.get(method, "unknown"),
            "n_chunks": int(src.n_chunks),
            "n_valid_chunks": len(values),
            "mean_chunk_length_chars": float(src.mean_chunk_length_chars),
        }
        moment_rows.append({
            **common, "moment_estimator": "raw", **raw_stats,
        })
        hist = histogram_frequency(values)
        for row in hist.to_dict("records"):
            frequency_rows.append({
                "source_id": src.source_id, "source_name": src.source_name,
                "method": method, "method_name": METHOD_NAMES.get(method, method),
                **row,
            })
        binned_stats = moments_from_histogram(hist)
        moment_rows.append({
            **common, "moment_estimator": "binned_fd_capped", **binned_stats,
        })

source_moments_df = pd.DataFrame(moment_rows)
frequency_df = pd.DataFrame(frequency_rows)

raw_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_source_moments_raw.csv"
binned_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_source_moments_binned_fd.csv"
freq_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_frequency_distributions_fd.csv"
source_moments_df[source_moments_df["moment_estimator"].eq("raw")].to_csv(raw_path, index=False, encoding="utf-8-sig")
source_moments_df[source_moments_df["moment_estimator"].eq("binned_fd_capped")].to_csv(binned_path, index=False, encoding="utf-8-sig")
frequency_df.to_csv(freq_path, index=False, encoding="utf-8-sig")
print("Raw source moments:", raw_path)
print("Binned source moments:", binned_path)
print("Frequency distributions:", freq_path)
display(source_moments_df.head())

Raw source moments: /kaggle/working/poet_compression_baseline/poet_source_moments_raw.csv
Binned source moments: /kaggle/working/poet_compression_baseline/poet_source_moments_binned_fd.csv
Frequency distributions: /kaggle/working/poet_compression_baseline/poet_frequency_distributions_fd.csv


,source_id,source_name,method,method_name,method_type,n_chunks,n_valid_chunks,mean_chunk_length_chars,moment_estimator,n_histogram_bins,mean_bpc,variance_bpc,skewness_bpc,excess_kurtosis_bpc,third_central_moment_bpc,fourth_central_moment_bpc
0,Amanda Gorman,Amanda Gorman,gpt2,GPT-2,language_model,146,144,686.842466,raw,NaN,3.135864,4.066380,0.941730,-0.748403,7.722153,37.231161
1,Amanda Gorman,Amanda Gorman,gpt2,GPT-2,language_model,146,144,686.842466,binned_fd_capped,8.0,3.171652,3.766763,0.984687,-0.686090,7.198646,32.830917
2,Amanda Gorman,Amanda Gorman,pythia_410m,Pythia-410M,language_model,146,144,686.842466,raw,NaN,2.869716,3.500794,0.886896,-0.924572,5.809284,25.435524
3,Amanda Gorman,Amanda Gorman,pythia_410m,Pythia-410M,language_model,146,144,686.842466,binned_fd_capped,8.0,2.882497,3.433052,0.897731,-0.929862,5.710405,24.398330
4,Amanda Gorman,Amanda Gorman,qwen2_5_0_5b,Qwen2.5-0.5B,language_model,146,144,686.842466,raw,NaN,2.874757,3.694706,0.959121,-0.697549,6.811505,31.430414


## 10. Compare the compression baselines with language models

In [15]:
from itertools import combinations

def safe_spearman(x, y, min_n=3):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < min_n or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan
    value = spearmanr(x, y).statistic
    return float(value) if np.isfinite(value) else np.nan

def mean_finite(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(values.mean()) if len(values) else np.nan

successful_lms = [m for m in MODEL_NAMES if m in set(source_moments_df["method"])]
if len(successful_lms) < 2:
    raise RuntimeError("At least two successful language models are required.")

correlation_rows = []
summary_rows = []
partial_rows = []
length_rows = []
coupling_rows = []

for estimator in ["raw", "binned_fd_capped"]:
    est_df = source_moments_df[source_moments_df["moment_estimator"].eq(estimator)]
    for measure in MEASURES:
        wide = est_df.pivot_table(index="source_id", columns="method", values=measure, aggfunc="first")
        # Pairwise correlations as reported in the paper: all LM-LM pairs.
        lm_pair_list = list(combinations(successful_lms, 2))
        for a, b in lm_pair_list:
            pair = wide.reindex(columns=[a, b]).dropna()
            rho = safe_spearman(pair[a], pair[b]) if not pair.empty else np.nan
            correlation_rows.append({
                "moment_estimator": estimator, "measure": measure,
                "comparison_type": "LM-LM", "method_1": a, "method_2": b,
                "n_sources": len(pair), "spearman_rho": rho,
            })

        # Compressor-to-LM correlations.
        for comp_method in COMPRESSION_METHODS:
            if comp_method not in wide.columns:
                continue
            for lm_method in successful_lms:
                pair = wide.reindex(columns=[comp_method, lm_method]).dropna()
                rho = safe_spearman(pair[comp_method], pair[lm_method]) if not pair.empty else np.nan
                correlation_rows.append({
                    "moment_estimator": estimator, "measure": measure,
                    "comparison_type": "compressor-LM", "method_1": comp_method,
                    "method_2": lm_method, "n_sources": len(pair), "spearman_rho": rho,
                })

        # Paired bootstrap on delta = mean(LM-LM rho) - mean(compressor-LM rho).
        for comp_method in COMPRESSION_METHODS:
            required = successful_lms + [comp_method]
            if not all(m in wide.columns for m in required):
                continue
            matched = wide[required].dropna()
            n = len(matched)
            point_lm_rhos = [
                safe_spearman(matched[a].to_numpy(), matched[b].to_numpy())
                for a, b in lm_pair_list
            ]
            point_comp_rhos = [
                safe_spearman(matched[comp_method].to_numpy(), matched[lm].to_numpy())
                for lm in successful_lms
            ]
            lm_mean_rho = mean_finite(point_lm_rhos)
            comp_mean_rho = mean_finite(point_comp_rhos)
            delta = lm_mean_rho - comp_mean_rho if np.isfinite(lm_mean_rho) and np.isfinite(comp_mean_rho) else np.nan

            boot_deltas = []
            if n >= 4:
                matrix = matched.to_numpy(dtype=float)
                col_idx = {col: j for j, col in enumerate(required)}
                boot_rng = np.random.default_rng(BOOTSTRAP_SEED)
                for _ in range(BOOTSTRAP_REPLICATES):
                    idx = boot_rng.integers(0, n, size=n)
                    sample = matrix[idx, :]
                    lm_rs = [
                        safe_spearman(sample[:, col_idx[a]], sample[:, col_idx[b]])
                        for a, b in lm_pair_list
                    ]
                    comp_rs = [
                        safe_spearman(sample[:, col_idx[comp_method]], sample[:, col_idx[lm]])
                        for lm in successful_lms
                    ]
                    av_lm = mean_finite(lm_rs)
                    av_comp = mean_finite(comp_rs)
                    if np.isfinite(av_lm) and np.isfinite(av_comp):
                        boot_deltas.append(av_lm - av_comp)

            lower = float(np.percentile(boot_deltas, 2.5)) if len(boot_deltas) >= 1000 else np.nan
            upper = float(np.percentile(boot_deltas, 97.5)) if len(boot_deltas) >= 1000 else np.nan
            summary_rows.append({
                "moment_estimator": estimator, "measure": measure,
                "compressor_method": comp_method,
                "n_sources": n,
                "lm_lm_mean_rho": lm_mean_rho,
                "compressor_lm_mean_rho": comp_mean_rho,
                "delta_lm_lm_minus_compressor_lm": delta,
                "delta_ci_95_lower": lower,
                "delta_ci_95_upper": upper,
                "valid_bootstrap_replicates": len(boot_deltas),
                "mean_rho_threshold": PRE_REGISTERED_MEAN_RHO_THRESHOLD if measure == "mean_bpc" else np.nan,
                "mean_threshold_met": bool(comp_mean_rho >= PRE_REGISTERED_MEAN_RHO_THRESHOLD) if measure == "mean_bpc" and np.isfinite(comp_mean_rho) else np.nan,
                "delta_ci_excludes_zero": bool(lower > 0 or upper < 0) if np.isfinite(lower) and np.isfinite(upper) else np.nan,
            })

        # Source-level coupling of mean and variance per method (length/mean confound checks).
        for method in successful_lms + COMPRESSION_METHODS:
            if method not in wide.columns:
                continue
            pair = est_df[est_df["method"].eq(method)].drop_duplicates("source_id").set_index("source_id")
            if "mean_bpc" in pair and "variance_bpc" in pair:
                valid = pair[["mean_bpc", "variance_bpc", "mean_chunk_length_chars", "n_valid_chunks"]].apply(pd.to_numeric, errors="coerce").dropna()
                if not valid.empty:
                    coupling_rows.append({
                        "moment_estimator": estimator, "method": method,
                        "n_sources": len(valid),
                        "pearson_mean_variance": float(pearsonr(valid["mean_bpc"], valid["variance_bpc"]).statistic) if valid["mean_bpc"].nunique() > 1 and valid["variance_bpc"].nunique() > 1 else np.nan,
                        "spearman_mean_variance": safe_spearman(valid["mean_bpc"], valid["variance_bpc"]),
                    })
                    length_rows.append({
                        "moment_estimator": estimator, "method": method,
                        "n_sources": len(valid),
                        "spearman_variance_vs_mean_chunk_length": safe_spearman(valid["variance_bpc"], valid["mean_chunk_length_chars"]),
                        "spearman_variance_vs_n_chunks": safe_spearman(valid["variance_bpc"], valid["n_valid_chunks"]),
                    })

    # Partial Spearman for variance controlling for each method's own mean rank.
    measure = "variance_bpc"
    wide_var = est_df.pivot_table(index="source_id", columns="method", values=measure, aggfunc="first")
    wide_mean = est_df.pivot_table(index="source_id", columns="method", values="mean_bpc", aggfunc="first")

    def partial_spearman_two_controls(x, y, control_x, control_y):
        arr = np.column_stack([x, y, control_x, control_y]).astype(float)
        arr = arr[np.all(np.isfinite(arr), axis=1)]
        if len(arr) < 5 or any(np.unique(arr[:, i]).size < 2 for i in range(4)):
            return np.nan
        rx, ry, rcx, rcy = [rankdata(arr[:, i]) for i in range(4)]
        design = np.column_stack([np.ones(len(arr)), rcx, rcy])
        rx_resid = rx - design @ np.linalg.lstsq(design, rx, rcond=None)[0]
        ry_resid = ry - design @ np.linalg.lstsq(design, ry, rcond=None)[0]
        if np.std(rx_resid) == 0 or np.std(ry_resid) == 0:
            return np.nan
        return float(np.corrcoef(rx_resid, ry_resid)[0, 1])

    all_methods = successful_lms + COMPRESSION_METHODS
    for a, b in combinations(all_methods, 2):
        if a not in wide_var or b not in wide_var or a not in wide_mean or b not in wide_mean:
            continue
        cols = pd.concat([
            wide_var[[a, b]].rename(columns={a:"var_a", b:"var_b"}),
            wide_mean[[a, b]].rename(columns={a:"mean_a", b:"mean_b"}),
        ], axis=1).dropna()
        if len(cols) < 5:
            continue
        if (a in successful_lms and b in successful_lms) or (a in COMPRESSION_METHODS and b in COMPRESSION_METHODS):
            continue
        rho_partial = partial_spearman_two_controls(
            cols["var_a"].to_numpy(), cols["var_b"].to_numpy(),
            cols["mean_a"].to_numpy(), cols["mean_b"].to_numpy(),
        )
        partial_rows.append({
            "moment_estimator": estimator,
            "method_1": a, "method_2": b,
            "n_sources": len(cols),
            "partial_spearman_variance_controlling_means": rho_partial,
        })

corr_df = pd.DataFrame(correlation_rows)
summary_df = pd.DataFrame(summary_rows)
partial_df = pd.DataFrame(partial_rows)
length_df = pd.DataFrame(length_rows).drop_duplicates()
coupling_df = pd.DataFrame(coupling_rows).drop_duplicates()

corr_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_pairwise_lm_and_compressor_correlations.csv"
summary_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_baseline_vs_lm_summary.csv"
partial_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_variance_partial_spearman.csv"
length_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_variance_length_confound_checks.csv"
coupling_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_mean_variance_coupling.csv"

corr_df.to_csv(corr_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
partial_df.to_csv(partial_path, index=False, encoding="utf-8-sig")
length_df.to_csv(length_path, index=False, encoding="utf-8-sig")
coupling_df.to_csv(coupling_path, index=False, encoding="utf-8-sig")

print("Saved pairwise correlations:", corr_path)
print("Saved LM-LM vs compressor-LM summaries:", summary_path)
print("Saved partial Spearman variance checks:", partial_path)
print("Saved variance/length confound checks:", length_path)
print("Saved mean-variance coupling:", coupling_path)
display(summary_df)

Saved pairwise correlations: /kaggle/working/poet_compression_baseline/poet_pairwise_lm_and_compressor_correlations.csv
Saved LM-LM vs compressor-LM summaries: /kaggle/working/poet_compression_baseline/poet_baseline_vs_lm_summary.csv
Saved partial Spearman variance checks: /kaggle/working/poet_compression_baseline/poet_variance_partial_spearman.csv
Saved variance/length confound checks: /kaggle/working/poet_compression_baseline/poet_variance_length_confound_checks.csv
Saved mean-variance coupling: /kaggle/working/poet_compression_baseline/poet_mean_variance_coupling.csv


,moment_estimator,measure,compressor_method,n_sources,lm_lm_mean_rho,compressor_lm_mean_rho,delta_lm_lm_minus_compressor_lm,delta_ci_95_lower,delta_ci_95_upper,valid_bootstrap_replicates,mean_rho_threshold,mean_threshold_met,delta_ci_excludes_zero
0,raw,mean_bpc,plain_deflate,31,0.961932,0.509383,0.452549,0.151101,0.801906,5000,0.85,False,True
1,raw,mean_bpc,plain_lzma,31,0.961932,0.526332,0.435600,0.157601,0.772512,5000,0.85,False,True
2,raw,mean_bpc,plain_bz2,31,0.961932,0.588378,0.373554,0.174513,0.656500,5000,0.85,False,True
3,raw,mean_bpc,plain_zstd,31,0.961932,0.579298,0.382634,0.167610,0.680258,5000,0.85,False,True
4,raw,mean_bpc,primed_deflate,31,0.961932,0.768967,0.192965,0.062694,0.417544,5000,0.85,False,True
5,raw,mean_bpc,primed_lzma,31,0.961932,0.800242,0.161690,0.046952,0.382067,5000,0.85,False,True
6,raw,variance_bpc,plain_deflate,31,0.952381,0.799637,0.152744,0.060233,0.325856,5000,NaN,NaN,True
7,raw,variance_bpc,plain_lzma,31,0.952381,0.803168,0.149213,0.065714,0.307931,5000,NaN,NaN,True
8,raw,variance_bpc,plain_bz2,31,0.952381,0.754237,0.198144,0.087642,0.403583,5000,NaN,NaN,True
9,raw,variance_bpc,plain_zstd,31,0.952381,0.781275,0.171106,0.072479,0.354456,5000,NaN,NaN,True


## Outputs and interpretation

The notebook computes both **raw moments from observed chunk scores** and **binned moments from capped Freedman–Diaconis frequency histograms**. The raw moments are the direct empirical moments of the chunk-level BPC values; the binned moments are a sensitivity analysis for the paper's histogram-based implementation.

Primary compression methods are raw Deflate and raw LZMA2, plus their reference-primed variants. BZip2 and Zstandard are reported as unprimed robustness checks. Compression BPC divides by `len(chunk)` in characters, not UTF-8 byte count. Primed scores are conditional size differences and are not clipped if a compressor returns a nonpositive difference.

The comparison summary reports, for each measure and estimator, the mean of the six available LM–LM Spearman correlations, the mean compressor–LM correlation, and their difference with a 95% source-level paired-bootstrap interval. The pre-specified descriptive threshold for a nontrivial mean-BPC baseline is 0.85 for the average compressor–LM correlation. This is an operational threshold, not a universal statistical standard.

Additional CSVs report pairwise correlations, partial Spearman correlations for variance while controlling for both methods' mean rankings, mean–variance coupling, and correlations of variance with mean chunk length and chunk count. These are diagnostics, not proof that compression and language-model scores measure the same underlying construct.